# 第22章：研究方法论

先计算计划内容的哈希，再保留含未定义尝试的六项试验账本，对五个有限p值作多重调整。随后比较32个变体，并分别模拟60个零假设世界和60个备择世界。内存中的哈希不是外部事前注册；French的24变体和历史图库默认跳过。结果主要保留在内存，创建输出目录不等于保存结果文件。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。运行步骤、数据要求和预期输出见本指南。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch22_research_methodology_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. 计划内容与哈希

规范化JSON生成计划内容哈希。只有与独立可信的先前摘要或记录比较，匹配才支持内容一致；单个现算哈希不能证明未改，更不能证明计划写在看数据之前。本例只在内存演示内容身份，没有外部事前注册。

In [ ]:
import dataclasses
from datetime import datetime, timezone
from quant_course.research_protocol import ResearchPlan, freeze_plan

plan = ResearchPlan(
    hypothesis="Cheap stocks (high B/M) earn higher next-month returns (sign +)",
    universe="constructed 200 names x 180 months",
    data_sources=("constructed panel, seed 20261006",),
    discovery_start="2000-01", discovery_end="2009-12", holdout_start="2010-01", holdout_end="2014-12",
    signal_definition="percentile rank of B/M within month, no lag",
    label_definition="next-month simple return",
    portfolio_rule="top quintile minus bottom quintile, equal weight",
    benchmark="equal-weight universe",
    primary_metric="t of mean IC: Pearson correlation of within-month signal ranks with next-month return, Newey-West lag 0, holdout",
    decision_rule="holdout t >= 2.0 and positive sign",
    cost_assumption="10 bp per unit traded", trial_budget=1)
clock = datetime(2026, 10, 6, 9, 0, tzinfo=timezone.utc)
frozen = freeze_plan(plan, now=clock)
edited = freeze_plan(dataclasses.replace(plan, decision_rule="holdout t >= 1.5 and positive sign"), now=clock)
print(frozen.sha256[:16], edited.sha256[:16])
assert frozen.sha256.startswith("40d38a74d67e0c08") and edited.sha256 != frozen.sha256

## 2. 试验账本与Bonferroni、Holm、BH

五个变体的$t=(2.4,1.1,0.4,1.9,2.1)$。只看最佳原始$p=0.0164$会低估选择作用；纳入五次有限检验后三种调整均未达到5%显著性，调整后p值大于0.05。第六个“未定义”尝试也保留在账本中。

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from quant_course.research_protocol import TrialLedger, benjamini_hochberg_adjust, bonferroni_adjust, holm_adjust

t = np.array([2.4, 1.1, 0.4, 1.9, 2.1])
ledger = TrialLedger()
for name, value in zip("abcde", t):
    ledger.record({"variant": name}, metric=float(value), n_obs=120, status="primary" if name == "a" else "run")
ledger.record({"variant": "f"}, metric=float("nan"), n_obs=0, status="abandoned", note="constant signal")
assert ledger.n_trials == 6
p = 2 * stats.norm.sf(t)
table = pd.DataFrame({"t": t, "p": p, "bonferroni": bonferroni_adjust(p), "holm": holm_adjust(p),
                      "bh": benjamini_hochberg_adjust(p)}, index=list("abcde"))
np.testing.assert_allclose(table["holm"].to_numpy(), [0.0820, 0.5427, 0.6892, 0.1723, 0.1429], atol=1e-4)
table.round(4)

## 3. $N$ 个零假设试验的最大值期望

DSR 近似式对小 $N$ 偏差大，对大 $N$ 好。$N=2$ 的精确值是 $1/\sqrt\pi$。

In [ ]:
from quant_course.research_protocol import expected_max_standard_normal, expected_max_standard_normal_exact

rows = [(n, expected_max_standard_normal(n), expected_max_standard_normal_exact(n)) for n in (2, 5, 10, 32, 100, 1000)]
emax = pd.DataFrame(rows, columns=["N", "approx", "exact"]).set_index("N")
emax["rel_error_%"] = 100 * (emax["approx"] / emax["exact"] - 1)
assert abs(emax.loc[2, "exact"] - 1 / np.sqrt(np.pi)) < 1e-9
emax.round(4)

## 4. 一个构造零世界：32 个变体，选最大 $|t|$

特征与收益相互独立，真实 IC 为 0。下面生成一个世界，计算 32 个变体在发现样本与留出样本上的 $t$，看挑出的最大值与留出结果。

In [ ]:
ch22 = load_code_module("ch22_research_methodology.py")
rng = np.random.default_rng(ch22.SEED)
x, r = ch22.simulate_world(rng, slope=0.0)
t_disc, t_hold, ic_disc, ic_hold = ch22.variant_t_stats(x, r)
variants = pd.DataFrame(ch22.VARIANTS, columns=["transform", "signal_lag", "horizon"])
variants["t_discovery"], variants["t_holdout"] = t_disc, t_hold
best = variants["t_discovery"].abs().idxmax()
print("registered primary (variant 0): discovery t =", round(t_disc[0], 2))
print("best of 32 by |discovery t|:", variants.loc[best, ["transform", "signal_lag", "horizon"]].tolist(), round(t_disc[best], 2),
      "-> holdout t =", round(t_hold[best], 2))
assert len(variants) == 32
variants.reindex(variants["t_discovery"].abs().sort_values(ascending=False).index).head(5).round(2)

## 5. 蒙特卡洛：零世界与备择世界

每个世界 60 次（正文用 500 次；本 Notebook 为了速度用 60，数字会有抽样波动）。固定种子，结果可复现。

In [ ]:
null = ch22.mining_experiment(0.0, 60, ch22.SEED)
alt = ch22.mining_experiment(ch22.ALT_SLOPE, 60, ch22.SEED + 1)
summary = pd.DataFrame({
    "null world": [null["primary_reject_rate_abs_t_gt_1.96"], null["best_of_32_reject_rate_abs_t_gt_1.96"],
                   null["best_of_32_reject_rate_bonferroni_hurdle"], null["selected_holdout_signed_t_gt_1.96_rate"]],
    "true IC 0.02": [alt["primary_reject_rate_abs_t_gt_1.96"], alt["best_of_32_reject_rate_abs_t_gt_1.96"],
                     alt["best_of_32_reject_rate_bonferroni_hurdle"], alt["selected_holdout_signed_t_gt_1.96_rate"]],
}, index=["primary |t|>1.96", "best-of-32 |t|>1.96", "best-of-32 |t|>3.16", "selected: holdout signed t>1.96"])
assert summary.loc["best-of-32 |t|>1.96", "null world"] > summary.loc["primary |t|>1.96", "null world"]
summary.round(3)

## 6. 默认禁用的French历史方案

历史本地计划固定发现样本1926年7月至1999年12月、留出样本2000年1月至2026年7月；本地哈希并非外部预注册。旧报告的留出$t=1.70$低于预先写下的2.0，属于历史结果，不由当前默认单元重算。

In [ ]:
if RUN_FRENCH:
    from quant_course.french_frozen import load_industries_and_rf
    
    returns, _, provenance = load_industries_and_rf()
    real_plan = ch22.real_plan()
    search = ch22.industry_search(returns, real_plan)
    frame = pd.DataFrame(search["table"])
    print("ledger rows:", search["ledger_trials"], "| French is point-in-time:", provenance["is_point_in_time"])
    primary = frame[frame["primary"]].iloc[0]
    assert (int(primary["n_disc"]), int(primary["n_hold"])) == (871, 319)
    assert abs(primary["t_disc"] - 4.8232) < 1e-3 and abs(primary["t_hold"] - 1.6993) < 1e-3
    assert search["n_variants_holdout_t_gt_2"] == 0
    frame.sort_values("t_disc", ascending=False).head(6).round(4)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 7. 自己试一试

1. 把 `decision_rule` 改成“留出 $t\ge1.5$”重算哈希；再说出为什么在看到 1.70 之后这样改会破坏方法。
2. 在第 4 节把 `slope` 改成 `ch22.ALT_SLOPE`，比较发现与留出 $t$。
3. 把第 2 节的 $M$ 改成 6（把未定义尝试也计入）并重算调整后 p 值。